# task_10 Solution

In [1]:

from pathlib import Path
import pandas as pd

base = Path("../../tasks/task_10/data/")


In [2]:

depots = pd.read_csv(base / "depots.csv")
scans = pd.read_csv(base / "scans.csv")
shipments = pd.read_csv(base / "shipments.csv")

Q1: For shipments weghing more than 20 kgs, which carrier has the highest on-time delivery rate?

In [3]:
shipments["ship_date"] = pd.to_datetime(shipments["ship_date"])
scans["scan_time"] = pd.to_datetime(scans["scan_time"])

first_events = scans.sort_values("scan_time").groupby(["shipment_id", "status"])["scan_time"].first().unstack()
ship = shipments.merge(first_events, on="shipment_id")
ship["delay_days"] = (ship["delivered"].dt.floor("D") - ship["ship_date"]).dt.days - ship["promised_days"]
ship["on_time"] = ship["delay_days"] <= 0

q1 = str(ship[ship["weight_kg"] > 20].groupby("carrier")["on_time"].mean().sort_values(ascending=False).index[0])
print(q1)

CR1


Q2: For shipments originating from South region depots, what is the median number of hours between picking up and recieving at the hub?

In [4]:
south_depots = set(depots[depots["region"] == "South"]["depot_id"])
south_ship = ship[ship["origin_depot"].isin(south_depots)].copy()
south_ship["pickup_to_hub_hours"] = (south_ship["hub_received"] - south_ship["picked_up"]).dt.total_seconds() / 3600
q2 = str(south_ship["pickup_to_hub_hours"].median())
print(q2)

12.5


Q4: What percentage of shipments with exactly 2 scans between transit were delivered late, rounded to 1 decimal place?

In [6]:
in_transit_counts = scans[scans["status"] == "in_transit"].groupby("shipment_id").size()
exactly_two = set(in_transit_counts[in_transit_counts == 2].index)
subset = ship[ship["shipment_id"].isin(exactly_two)]
q4 = f"{(100 * (subset['delay_days'] > 0).mean()):.1f}%"
print(q4)

50.0%
